In [3]:
experiment = "vggnet16_benchmark2022_segmented_one_img"

In [4]:
# To sort the results.csv by instance_id
CSV_PATH = f"../results/{experiment}/results.csv"

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv(CSV_PATH)

# --- 1) Filter GLOBAL rows ---
# Prefer the explicit boolean if present; otherwise fall back to tag/pattern
if "is_global" in df.columns:
    d = df[df["is_global"] == True].copy()
else:
    d = df[
        df["tag"].str.contains("global", case=False, na=False)
        | df.get("pattern", pd.Series("", index=df.index)).astype(str).str.contains("global", case=False, na=False)
    ].copy()

# --- 2) Pick a time column ---
TIME_COL = "all_time" if "all_time" in d.columns else "bab_time"
d[TIME_COL] = pd.to_numeric(d[TIME_COL], errors="coerce")

# --- 3) Aggregate time by epsilon (mean + median are both useful) ---
time_eps = (
    d.dropna(subset=["eps", TIME_COL])
     .groupby("eps")[TIME_COL]
     .agg(n="count", mean="mean", median="median", std="std")
     .reset_index()
     .sort_values("eps")
)

display(time_eps)

# --- 4) Plot ---
plt.figure()
plt.plot(time_eps["eps"], time_eps["mean"], marker="o", label="mean")
plt.plot(time_eps["eps"], time_eps["median"], marker="o", label="median")
plt.xlabel("epsilon")
plt.ylabel(TIME_COL)
plt.title(f"GLOBAL images: {TIME_COL} vs epsilon")
plt.legend()
plt.show()

KeyError: 'tag'